# 05 — Scenario Simulation & Monte Carlo Risk

## Portfolio objective

Stress-test the optimized capacity plan under deterministic shocks and probabilistic uncertainty.

### Deterministic scenarios

- Base
- High demand +10%
- Demand surge +25%
- Capacity outage −15%
- Combined stress: +20% requirement / −15% capacity

### Monte Carlo

5,000 simulations using stochastic requirement and capacity multipliers.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from cpoe.config import PROCESSED, FIGURES
from cpoe.simulation.scenarios import (
    run_deterministic_scenarios,
    run_monte_carlo,
)

capacity = pd.read_csv(
    PROCESSED / "capacity_requirements.csv",
    index_col=0,
    parse_dates=True,
)
solution = pd.read_csv(
    PROCESSED / "optimization_solution.csv",
    index_col=0,
    parse_dates=True,
)

scenario_summary = run_deterministic_scenarios(
    capacity["required_capacity"],
    solution["available_capacity"],
)
scenario_summary.to_csv(PROCESSED / "scenario_summary.csv", index=False)

mc_results, mc_summary = run_monte_carlo(
    capacity["required_capacity"],
    solution["available_capacity"],
)
mc_results.to_csv(PROCESSED / "monte_carlo_results.csv", index=False)
mc_summary.to_csv(PROCESSED / "monte_carlo_risk_summary.csv")

display(scenario_summary)
display(mc_summary)


In [ ]:
plt.figure(figsize=(12, 5))
for scenario in scenario_summary["scenario"]:
    # Reconstruct the stress-test shortage rate as a compact comparison.
    row = scenario_summary.loc[scenario_summary["scenario"] == scenario].iloc[0]
    plt.bar(scenario, row["shortage_rate"])

plt.title("Scenario Shortage Rate")
plt.xlabel("Scenario")
plt.ylabel("Shortage rate")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(FIGURES / "scenario_shortage_rate.png", dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(mc_results["peak_shortage"], bins=40)
plt.title("Monte Carlo Peak Shortage Distribution")
plt.xlabel("Peak shortage")
plt.ylabel("Simulation count")
plt.tight_layout()
plt.savefig(FIGURES / "monte_carlo_shortage_distribution.png", dpi=150)
plt.show()


## Final decision framework

1. Server trace analysis
2. Capacity-footprint forecasting
3. Capacity planning and headroom
4. Integer capacity expansion optimization
5. Deterministic stress testing
6. Monte Carlo risk assessment

The project therefore demonstrates an end-to-end capacity-planning decision workflow rather than a standalone forecasting model.
